<a href="https://colab.research.google.com/github/rokosu/Going-Concern-/blob/main/table5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**As part of Research Paper:**                          
                 ***📊 Going Concern Uncertainties & Financial Statement Quality
Empirical Analysis of Management vs. Auditors' Disclosures.

Table 5 (LASSO-logistic regression results for distress prediction models

Author: Johnson-Rokosu, Samuel F. | Chartered Accountant | Forensic Accounting Researcher | Python for Financial Analytics


In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
import statsmodels.api as sm
import warnings

# Suppress overflow warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# --------------------------------------------
# Step 1: Generate Synthetic Data
# --------------------------------------------
np.random.seed(42)
n_samples = 1000

data = pd.DataFrame({
    'Altman_Z': np.random.uniform(1.0, 3.0, n_samples),
    'Liquidity_Ratio': np.random.uniform(0.5, 2.0, n_samples),
    'Debt_Covenant_Breaches': np.random.randint(0, 3, n_samples),
    'ESG_Score': np.random.uniform(0, 1, n_samples),
})
data['Bankruptcy_Within_12Months'] = np.where(data['Altman_Z'] < 1.8, 1, 0)

# --------------------------------------------
# Step 2: Preprocess Data
# --------------------------------------------
X = data.drop('Bankruptcy_Within_12Months', axis=1)
y = data['Bankruptcy_Within_12Months']
X = X.fillna(X.mean())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# --------------------------------------------
# Step 3: LASSO-Logistic Regression
# --------------------------------------------
lasso_logit = LogisticRegression(penalty='l1', solver='liblinear', C=0.1)
lasso_logit.fit(X_train, y_train)

coef = lasso_logit.coef_[0]
selected_features = np.where(coef != 0)[0]

# Ensure features are selected
if len(selected_features) == 0:
    raise ValueError("No features selected by LASSO. Decrease regularization (increase C).")

X_selected = X.iloc[:, selected_features]

# Check for multicollinearity
corr_matrix = X_selected.corr().abs()
np.fill_diagonal(corr_matrix.values, 0)
if (corr_matrix > 0.9).any().any():
    print("Warning: High multicollinearity. Remove collinear features.")

# --------------------------------------------
# Step 4: Refit with Statsmodels (No Intercept)
# --------------------------------------------
X_sm = X_selected.copy()  # No intercept to avoid singularity
logit_model = sm.Logit(y, X_sm)
result = logit_model.fit(maxiter=100)  # Increase iterations to prevent warnings

# --------------------------------------------
# Step 5: Generate Table 5
# --------------------------------------------
# Coefficients and p-values
statsmodels_coef = result.params.values
p_values = result.pvalues.values
conf_int = result.conf_int().values

table5 = pd.DataFrame({
    'Predictor': X_selected.columns.tolist(),
    'Coefficient': np.round(statsmodels_coef, 3),
    'P-value': np.round(p_values, 3),
    'Lower CI': np.round(conf_int[:, 0], 3),
    'Upper CI': np.round(conf_int[:, 1], 3)
})

# Performance metrics
y_pred = lasso_logit.predict(X_test)
y_proba = lasso_logit.predict_proba(X_test)[:, 1]
auc = np.round(roc_auc_score(y_test, y_proba), 3)
accuracy = np.round(accuracy_score(y_test, y_pred), 3)

table5_footer = pd.DataFrame({
    'Predictor': ['Model Accuracy', 'AUC-ROC'],
    'Coefficient': [accuracy, auc],
    'P-value': [np.nan, np.nan],
    'Lower CI': [np.nan, np.nan],
    'Upper CI': [np.nan, np.nan]
})

table5 = pd.concat([table5, table5_footer], ignore_index=True)

# --------------------------------------------
# Final Output: Table 5
# --------------------------------------------
print("Table 5: LASSO-Logistic Regression Results")
print(table5.to_string(index=False))

Optimization terminated successfully.
         Current function value: 0.614052
         Iterations 5
Table 5: LASSO-Logistic Regression Results
     Predictor  Coefficient  P-value  Lower CI  Upper CI
      Altman_Z       -0.407      0.0    -0.474     -0.34
Model Accuracy        0.987      NaN       NaN       NaN
       AUC-ROC        1.000      NaN       NaN       NaN
